# PS2 Assignment Results

## Credit Default Prediction Results

**Final Model Performance:**
- Training Loss: 0.4419, Accuracy: 0.7896
- Validation Loss: 0.4635, Accuracy: 0.7767  
- Test Loss: 0.4560, Accuracy: 0.7764

**Test Set Metrics:**
- Test Accuracy: 77.64%
- Precision (Default): 0.81
- Recall (Default): 0.90
- F1-Score (Default): 0.85

In [ ]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Load test data
print("Loading test data...")
df = pd.read_parquet('testing_data_ps2_1.parquet')
print(f"Test data shape: {df.shape}")

# Separate features and target (if exists)
if 'default' in df.columns:
    X_test = df.drop('default', axis=1)
    y_test = df['default']
    print(f"Target found. Test labels shape: {y_test.shape}")
else:
    X_test = df
    y_test = None
    print("No target column found.")

# Load preprocessors
scaler = joblib.load('credit_scaler.pkl')
feature_names = joblib.load('feature_names.pkl')

# Handle missing values
for col in X_test.columns:
    if X_test[col].isnull().sum() > 0:
        if X_test[col].dtype in ['int64', 'float64']:
            X_test[col] = X_test[col].fillna(X_test[col].median())
        else:
            X_test[col] = X_test[col].fillna(X_test[col].mode()[0])

# Identify categorical and numerical columns
categorical_cols = X_test.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X_test.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numerical columns: {numerical_cols}")

# Create one-hot encoding for categorical variables
if categorical_cols:
    # Create a simple one-hot encoder
    ohe = OneHotEncoder(sparse_output=False, drop='first')
    categorical_data = ohe.fit_transform(X_test[categorical_cols])
    
    # Create column names for one-hot encoded features
    categorical_feature_names = []
    for i, col in enumerate(categorical_cols):
        categories = ohe.categories_[i][1:]  # Drop first category (drop='first')
        for cat in categories:
            categorical_feature_names.append(f"{col}_{cat}")
    
    # Create DataFrame with one-hot encoded features
    categorical_df = pd.DataFrame(categorical_data, columns=categorical_feature_names, index=X_test.index)
    
    # Combine numerical and categorical features
    X_test_encoded = pd.concat([X_test[numerical_cols], categorical_df], axis=1)
else:
    X_test_encoded = X_test

print(f"Encoded features shape: {X_test_encoded.shape}")
print(f"Expected features: {len(feature_names)}")

# Ensure all columns are present and in the right order
missing_cols = set(feature_names) - set(X_test_encoded.columns)
extra_cols = set(X_test_encoded.columns) - set(feature_names)

if missing_cols:
    print(f"Missing columns: {missing_cols}")
    # Add missing columns with zeros
    for col in missing_cols:
        X_test_encoded[col] = 0

if extra_cols:
    print(f"Extra columns: {extra_cols}")
    # Remove extra columns
    X_test_encoded = X_test_encoded.drop(columns=list(extra_cols))

# Ensure correct column order
X_test_encoded = X_test_encoded[feature_names]

# Scale features
X_test_scaled = scaler.transform(X_test_encoded.astype(float))
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names)

# Load model and predict
model = keras.models.load_model('credit_default_dnn_model.h5')
predictions_proba = model.predict(X_test_scaled)
predictions = (predictions_proba > 0.5).astype(int).flatten()

# Create results
results_df = pd.DataFrame({
    'prediction': predictions,
    'prediction_probability': predictions_proba.flatten()
})

if y_test is not None:
    accuracy = np.mean(predictions == y_test)
    print(f"\nTest Accuracy: {accuracy:.4f}")
    results_df['actual'] = y_test.values

# Save results
results_df.to_csv('test_predictions.csv', index=False)
predictions_df = pd.DataFrame({'prediction': predictions})
predictions_df.to_csv('predictions_only.csv', index=False, header=False)

print(f"Generated {len(predictions)} predictions")
print(f"Predictions saved to 'test_predictions.csv' and 'predictions_only.csv'")
print(f"Prediction distribution: Default={np.sum(predictions==1)} ({np.mean(predictions==1):.1%}), No Default={np.sum(predictions==0)} ({np.mean(predictions==0):.1%})")

Loading test data...
Test data shape: (41248, 26)
Target found. Test labels shape: (41248,)
Categorical columns: ['gender', 'marital_status', 'education_level', 'region', 'home_ownership', 'favorite_color', 'zodiac_sign', 'generation']
Numerical columns: ['age', 'income', 'credit_score', 'credit_history_length', 'num_credit_accounts', 'debt_to_income_ratio', 'loan_amount', 'employment_length', 'application_month', 'application_year', 'birth_month', 'monthly_income', 'annual_debt_payment', 'credit_utilization', 'income_rank_in_age_group', 'lucky_number', 'customer_id']
Encoded features shape: (41248, 55)
Expected features: 35
Missing columns: {'generation_Boomer'}
Extra columns: {'zodiac_sign_Leo', 'zodiac_sign_Pisces', 'favorite_color_Purple', 'favorite_color_White', 'favorite_color_Yellow', 'zodiac_sign_Capricorn', 'zodiac_sign_Virgo', 'zodiac_sign_Libra', 'zodiac_sign_Aries', 'favorite_color_Blue', 'generation_Gen Z/Millennial', 'zodiac_sign_Scorpio', 'favorite_color_Orange', 'zodiac

1289/1289 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step

Test Accuracy: 0.7852
Generated 41248 predictions
Predictions saved to 'test_predictions.csv' and 'predictions_only.csv'
Prediction distribution: Default=33512 (81.2%), No Default=7736 (18.8%)


## CNN Stock Return Prediction Results

### 1) CNN Models with Bollinger Bands Labels

**Window Size 5:**
- BB charts generated: 0 (insufficient data for BB calculation)
- No model could be trained

**Window Size 20:**
- BB charts generated: 1000 (312 buy, 688 sell)
- Training Accuracy: 99.57%
- Validation Accuracy: 99.00%
- **Test Accuracy: 96.00%**

### 2) CNN Models with Short-term Gains Labels

**Window Size 5:**
- Charts generated: 1000
- Training Accuracy: 61.14%
- Validation Accuracy: 59.00%
- **Test Accuracy: 63.00%**

**Window Size 20:**
- Charts generated: 1000
- Training Accuracy: 60.14%
- Validation Accuracy: 57.00%
- **Test Accuracy: 58.00%**

### 3) Why Random Split in Stock Forecasting is Problematic

Random splitting in stock forecasting creates several critical issues:

**Look-ahead Bias:**
- Random splits can put data from similar time periods in both training and test sets
- The model "sees" future information during training that wouldn't be available in real trading
- This explains why the BB model achieved unrealistic 96% accuracy

**Temporal Structure Violation:**
- Stock prices have time dependencies and autocorrelation
- Random splitting breaks these natural time relationships
- In real trading, you can only use past data to predict future returns

**Overfitting to Patterns:**
- The CNN memorizes visual chart patterns rather than learning market dynamics
- High training accuracy (99.57%) vs test accuracy (96%) suggests the model is overfitting
- These patterns wouldn't generalize to truly unseen future data

**Real-world Trading Failure:**
- The 96% accuracy would never be achievable in live trading
- Proper time-series validation (using only past data to predict future) would show much lower accuracy
- Random splits create false confidence that would lead to significant trading losses

**Proper Approach:**
- Use chronological splits where training data comes before test data
- Implement walk-forward analysis to simulate real trading conditions
- Test models on truly out-of-sample future periods